In [ ]:
# inlegalbert_kg_rag_llama_rrc.py  (KG-RAG + LLaMA LoRA ARCHITECTURE)
#
# 5-Layer Architecture:
#   [Layer 1] InLegalBERT → Sentence Embedding (with context window)
#   [Layer 2] KG-RAG Retrieval → triples → text
#   [Layer 3] Context Fusion (sentence emb + KG facts + positional features)
#   [Layer 4] LLaMA (LoRA fine-tuned) → chain-of-thought reasoning
#   [Layer 5] Linear → Softmax → Rhetorical Role Label
#
# Class Imbalance Handling:
#   - LDAM Loss + class-weighted CE auxiliary loss
#   - Focal Loss component for hard examples
#   - Rare-class always-KG retrieval
#   - Label smoothing
#   - Balanced sampling via WeightedRandomSampler
#
# Training:
#   Phase A – Train InLegalBERT + BiLSTM + MHA base encoder
#   Phase A→B – Build Knowledge Graph from train embeddings
#   Phase B – Fine-tune full pipeline with LoRA LLaMA head

import os, json, random, time, math
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import (
    AutoTokenizer, AutoModel,
    AutoModelForCausalLM, AutoTokenizer as LLMTokenizer,
    get_linear_schedule_with_warmup,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig, get_peft_model, TaskType,
    prepare_model_for_kbit_training,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
LLAMA_MODEL_NAME       = "meta-llama/Llama-2-7b-hf"   # or "meta-llama/Meta-Llama-3-8B"

TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_kg_rag_llama_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32      # per sentence for BERT
CONTEXT_WINDOW  = 3       # neighbouring sentences for context window
BATCH_DOCS      = 2
NUM_EPOCHS_BASE = 60      # Phase A
NUM_EPOCHS_KG   = 20      # Phase B
BERT_LR         = 1e-5
HEAD_LR         = 5e-4
LORA_LR         = 2e-4
WEIGHT_DECAY    = 0.05
GRAD_CLIP       = 1.0
DROPOUT         = 0.4

# BERT freeze
BERT_FREEZE_LAYERS = 8
BERT_LR_DECAY      = 0.9

# Sentence-level BiLSTM
SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 2

# Multi-Head Attention Pooling
MHA_HEADS   = 4
MHA_DROPOUT = 0.1

# Context-enrichment BiLSTM
CTX_LSTM_HIDDEN = 64
CTX_LSTM_LAYERS = 2

# Loss hyperparams
AUX_CE_WEIGHT    = 0.2
LABEL_SMOOTHING  = 0.1
FOCAL_GAMMA      = 2.0
LDAM_MAX_MARGIN  = 0.5

# Early stopping
ES_PATIENCE  = 10
ES_MIN_DELTA = 1e-4

WARMUP_RATIO = 0.05
GRADIENT_ACCUMULATION_STEPS = 2
RARE_THRESHOLD = 0.05

# KG-RAG
KG_TOP_K           = 3
KG_TOP_NODES       = 5
KG_HOP             = 1
UNCERTAINTY_THRESH = 0.7
RARE_ALWAYS_KG     = True
KG_FUSION_DIM      = 256   # sent_out_dim = SENT_LSTM_HIDDEN * 2
RST_INTRA_THRESH   = 0.6
RST_CROSS_THRESH   = 0.5
MAX_KG_TEXT_TOKENS = 128   # max tokens from KG facts for LLaMA

# LoRA config for LLaMA
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05
LORA_TARGET     = ["q_proj", "v_proj"]
USE_4BIT        = True     # QLoRA 4-bit quantisation; set False if no bitsandbytes

# LLaMA hidden size (depends on model; 7B = 4096, 8B = 4096)
LLAMA_HIDDEN    = 4096
FUSION_DIM      = 512      # projection dim before LLaMA

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"

# Label descriptions for LLaMA prompt
LABEL_DESCRIPTIONS = {
    "PREAMBLE":        "introductory section of the judgment",
    "FAC":             "facts of the case",
    "RLC":             "ruling by the lower court",
    "ISSUE":           "legal issue under consideration",
    "ARG_PETITIONER":  "arguments made by the petitioner",
    "ARG_RESPONDENT":  "arguments made by the respondent",
    "ANALYSIS":        "legal analysis and reasoning by the court",
    "STA":             "statute or law cited",
    "PRE_RELIED":      "precedent relied upon",
    "PRE_NOT_RELIED":  "precedent not relied upon",
    "RATIO":           "ratio decidendi / legal principle",
    "RPC":             "ruling by the present court",
    "NONE":            "none of the above categories",
}


# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -1.0
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))
        if not sents or len(sents) != len(labs):
            continue
        sents = sents[:max_sents]
        labs  = labs[:max_sents]
        all_docs.append((sents, labs))
    return all_docs


def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}
    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]
    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


def compute_class_weights(docs, rare_ids):
    """Effective number of samples class weighting for imbalance."""
    all_ids = [lid for _, labs in docs for lid in labs]
    counts  = Counter(all_ids)
    beta    = 0.9999
    weights = []
    for i in range(NUM_LABELS):
        n_i = counts.get(i, 1)
        eff = (1.0 - beta ** n_i) / (1.0 - beta)
        weights.append(1.0 / eff)
    weights = torch.tensor(weights, dtype=torch.float)
    weights = weights / weights.sum() * NUM_LABELS
    return weights


def compute_sample_weights(docs):
    """Per-sample weights for WeightedRandomSampler."""
    all_ids = [lid for _, labs in docs for lid in labs]
    counts  = Counter(all_ids)
    total   = len(all_ids)
    doc_weights = []
    for _, labs in docs:
        # weight each doc by the inverse frequency of its rarest label
        doc_w = max(total / max(counts.get(l, 1), 1) for l in labs)
        doc_weights.append(doc_w)
    return doc_weights


# ═══════════════════════════════════════════════════════════
# LOSSES
# ═══════════════════════════════════════════════════════════
class FocalLoss(nn.Module):
    def __init__(self, gamma=FOCAL_GAMMA, weight=None,
                 label_smoothing=LABEL_SMOOTHING, ignore_index=-100):
        super().__init__()
        self.gamma          = gamma
        self.weight         = weight
        self.label_smoothing = label_smoothing
        self.ignore_index   = ignore_index

    def forward(self, logits, targets):
        # logits: (N, C), targets: (N,)
        mask = targets != self.ignore_index
        logits  = logits[mask]
        targets = targets[mask]
        if logits.numel() == 0:
            return logits.sum() * 0

        log_p = F.log_softmax(logits, dim=-1)
        p     = log_p.exp()

        # Label smoothing
        C = logits.size(-1)
        smooth_loss = -log_p.mean(dim=-1)
        nll_loss    = F.nll_loss(log_p, targets,
                                 weight=self.weight.to(logits.device)
                                 if self.weight is not None else None,
                                 reduction="none")
        loss = (1.0 - self.label_smoothing) * nll_loss + \
               self.label_smoothing / C * smooth_loss

        pt    = p.gather(1, targets.unsqueeze(1)).squeeze(1)
        focal = (1.0 - pt) ** self.gamma * loss
        return focal.mean()


class LDAMLoss(nn.Module):
    """Label-Distribution-Aware Margin Loss."""
    def __init__(self, cls_num_list, max_m=LDAM_MAX_MARGIN,
                 weight=None, s=30):
        super().__init__()
        m_list = 1.0 / np.sqrt(np.sqrt(cls_num_list))
        m_list = m_list * (max_m / m_list.max())
        self.m_list = torch.tensor(m_list, dtype=torch.float)
        self.s      = s
        self.weight = weight

    def forward(self, logits, targets):
        mask = targets != -100
        logits  = logits[mask]
        targets = targets[mask]
        if logits.numel() == 0:
            return logits.sum() * 0

        index = torch.zeros_like(logits, dtype=torch.uint8)
        index.scatter_(1, targets.unsqueeze(1), 1)
        m = self.m_list.to(logits.device)
        index_float = index.float()
        batch_m = m[targets]
        logits_m = logits - index_float * batch_m.unsqueeze(1)
        output = self.s * logits_m
        return F.cross_entropy(output, targets,
                               weight=self.weight.to(logits.device)
                               if self.weight is not None else None)


class CombinedImbalanceLoss(nn.Module):
    """LDAM + Focal + CE combo for maximal minority-class performance."""
    def __init__(self, cls_num_list, class_weights,
                 ldam_weight=0.4, focal_weight=0.4, ce_weight=0.2):
        super().__init__()
        self.ldam  = LDAMLoss(cls_num_list, weight=class_weights)
        self.focal = FocalLoss(weight=class_weights)
        self.ce    = nn.CrossEntropyLoss(
            weight=class_weights,
            label_smoothing=LABEL_SMOOTHING,
            ignore_index=-100,
        )
        self.lw = ldam_weight
        self.fw = focal_weight
        self.cw = ce_weight

    def forward(self, logits, targets):
        return (self.lw * self.ldam(logits, targets) +
                self.fw * self.focal(logits, targets) +
                self.cw * self.ce(logits, targets))


# ═══════════════════════════════════════════════════════════
# DATASET  (with context window)
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH,
                 context_window=CONTEXT_WINDOW):
        self.docs           = docs
        self.tokenizer      = tokenizer
        self.max_length     = max_length
        self.context_window = context_window

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        T = len(sents)

        # Context-aware encoding: prepend/append neighbour sentences
        context_sents = []
        for t in range(T):
            lo = max(0, t - self.context_window)
            hi = min(T, t + self.context_window + 1)
            ctx = " [SEP] ".join(sents[lo:hi])
            context_sents.append(ctx)

        enc = self.tokenizer(
            context_sents,
            padding="max_length",
            truncation=True,
            max_length=self.max_length * (2 * self.context_window + 1),
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get(
                "token_type_ids",
                torch.zeros_like(enc["input_ids"])
            ),
            "labels":  torch.tensor(labels, dtype=torch.long),
            "raw_sents": sents,  # kept for KG text & LLaMA prompts
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L     = batch[0]["input_ids"].shape[1]
    B     = len(batch)

    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)
    raw_sents_batch = []

    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t
        raw_sents_batch.append(b["raw_sents"])

    return input_ids, attention_mask, token_type_ids, labels, lengths, raw_sents_batch


# ═══════════════════════════════════════════════════════════
# MULTI-HEAD ATTENTION POOLING
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads

        self.query = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)

        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)
        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x, key_padding_mask=None):
        N, L, H = x.shape
        K = self.key_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.val_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        w = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        if key_padding_mask is not None:
            w = w.masked_fill(key_padding_mask.unsqueeze(1).unsqueeze(2), -1e9)
        w = self.attn_drop(F.softmax(w, dim=-1))
        ctx = torch.matmul(w, V).squeeze(2).reshape(N, H)
        return self.out_proj(ctx)


# ═══════════════════════════════════════════════════════════
# LAYER 1: InLegalBERT ENCODER  (document-level with context window)
# ═══════════════════════════════════════════════════════════
class InLegalBERTEncoder(nn.Module):
    """
    Layer 1: InLegalBERT → sentence embedding using BiLSTM + MHA pooling.
    Each sentence is encoded with its context window already baked in
    at the tokenisation stage.
    """
    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
    ):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size    # 768
        self.dropout  = nn.Dropout(dropout)

        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size    = self.bert_dim,
            hidden_size   = sent_lstm_hidden,
            num_layers    = sent_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if sent_lstm_layers > 1 else 0.0,
        )
        self.sent_out_dim = sent_lstm_hidden * 2  # 256

        self.mha_pooling = MultiHeadAttentionPooling(
            hidden_dim = self.sent_out_dim,
            num_heads  = mha_heads,
            dropout    = mha_dropout,
        )
        self.sent_layer_norm = nn.LayerNorm(self.sent_out_dim)

    def _freeze_bert_layers(self, n_freeze):
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False
        for i in range(min(n_freeze, len(self.bert.encoder.layer))):
            for param in self.bert.encoder.layer[i].parameters():
                param.requires_grad = False
        n = len(self.bert.encoder.layer)
        print(f"❄️  BERT frozen: embeddings + layers 0-{n_freeze-1}")
        print(f"🔥 BERT trainable: layers {n_freeze}-{n-1} + pooler\n")

    def forward(self, input_ids, attention_mask, token_type_ids, lengths=None):
        """Returns sentence-level embeddings: (B, T, sent_out_dim)."""
        B, T, L = input_ids.shape
        N = B * T
        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)

        valid = flat_mask.sum(dim=-1) > 0
        token_embs = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)

        if valid.any():
            out = self.bert(
                input_ids      = flat_ids[valid],
                attention_mask = flat_mask[valid],
                token_type_ids = flat_types[valid],
            )
            token_embs[valid] = out.last_hidden_state.to(token_embs.dtype)

        token_embs = self.dropout(token_embs)
        lstm_out, _ = self.sent_bilstm(token_embs)
        lstm_out    = self.dropout(lstm_out)

        pad_mask = (flat_mask == 0).clone()
        pad_mask[~valid] = False

        sent_vecs = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs = self.sent_layer_norm(sent_vecs)
        sent_vecs = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)
        return sent_vecs.view(B, T, -1)   # (B, T, 256)


# ═══════════════════════════════════════════════════════════
# KNOWLEDGE GRAPH
# ═══════════════════════════════════════════════════════════
class KnowledgeGraph:
    def __init__(self, emb_dim=KG_FUSION_DIM):
        self.emb_dim     = emb_dim
        self.nodes       = defaultdict(list)
        self.intra_edges = defaultdict(list)
        self.cross_edges = []
        self._stacked    = {}

    def add_nodes(self, embeddings, label_ids, texts=None):
        embs = embeddings.detach().cpu()
        for k, (emb, lid) in enumerate(zip(embs, label_ids)):
            text = texts[k] if texts is not None else ""
            self.nodes[lid].append({"emb": emb, "text": text})
        self._stacked = {}

    def build_edges(self, intra_thresh=RST_INTRA_THRESH,
                    cross_thresh=RST_CROSS_THRESH,
                    max_intra_per_node=5, max_cross=2000):
        print("  Building KG edges ...")
        self._stacked = {}
        self.intra_edges = defaultdict(list)
        self.cross_edges = []

        for lid, node_list in self.nodes.items():
            N = len(node_list)
            if N < 2:
                continue
            embs = torch.stack([n["emb"] for n in node_list])
            enorm = F.normalize(embs, dim=-1)
            sim = torch.mm(enorm, enorm.T)
            for i in range(N - 1):
                self.intra_edges[lid].append((i, i+1, float(sim[i, i+1])))
            for i in range(N):
                s = sim[i].clone()
                s[max(0,i-1):i+2] = -1
                for _ in range(max_intra_per_node):
                    j = int(s.argmax())
                    if s[j] < intra_thresh:
                        break
                    self.intra_edges[lid].append((i, j, float(s[j])))
                    s[j] = -1
            self._stacked[lid] = embs

        label_ids = list(self.nodes.keys())
        cross_count = 0
        for a in range(len(label_ids)):
            if cross_count >= max_cross:
                break
            for b in range(a+1, len(label_ids)):
                if cross_count >= max_cross:
                    break
                la, lb = label_ids[a], label_ids[b]
                ea = self._get_stacked(la)
                eb = self._get_stacked(lb)
                if ea is None or eb is None:
                    continue
                sim = torch.mm(F.normalize(ea, dim=-1), F.normalize(eb, dim=-1).T)
                high = (sim >= cross_thresh).nonzero(as_tuple=False)
                for pair in high[:50]:
                    ni, nj = int(pair[0]), int(pair[1])
                    self.cross_edges.append((la, ni, lb, nj, float(sim[ni, nj])))
                    cross_count += 1

        n_intra = sum(len(v) for v in self.intra_edges.values())
        n_nodes = sum(len(v) for v in self.nodes.values())
        print(f"  KG: {n_nodes} nodes | {n_intra} intra | {len(self.cross_edges)} cross")

    def _get_stacked(self, lid):
        if lid not in self._stacked:
            if lid not in self.nodes or not self.nodes[lid]:
                return None
            self._stacked[lid] = torch.stack([n["emb"] for n in self.nodes[lid]])
        return self._stacked[lid]

    def retrieve_text_facts(self, h_i, top_k=KG_TOP_K, top_nodes=KG_TOP_NODES):
        """Return text snippets from top-K retrieved nodes."""
        h_norm = F.normalize(h_i.unsqueeze(0), dim=-1)
        scores = {}
        for lid in self.nodes:
            embs = self._get_stacked(lid)
            if embs is None:
                continue
            sims = torch.mv(F.normalize(embs, dim=-1), h_norm.squeeze(0))
            scores[lid] = float(sims.max())
        top_lids = sorted(scores, key=lambda x: -scores[x])[:top_k]
        facts = []
        for lid in top_lids:
            embs = self._get_stacked(lid)
            sims = torch.mv(F.normalize(embs, dim=-1), h_norm.squeeze(0))
            k    = min(top_nodes, embs.shape[0])
            idxs = sims.topk(k).indices.tolist()
            label_name = id2label[lid]
            for idx in idxs:
                text = self.nodes[lid][idx]["text"]
                if text.strip():
                    facts.append(f"[{label_name}]: {text}")
        return facts

    def save(self, path):
        data = {
            "nodes": {str(k): [{"emb": n["emb"].tolist(), "text": n["text"]}
                                for n in v]
                      for k, v in self.nodes.items()},
            "intra_edges": {str(k): v for k, v in self.intra_edges.items()},
            "cross_edges": self.cross_edges,
        }
        with open(path, "w") as f:
            json.dump(data, f)
        print(f"  KG saved to {path}")

    @classmethod
    def load(cls, path, emb_dim=KG_FUSION_DIM):
        kg = cls(emb_dim=emb_dim)
        with open(path) as f:
            data = json.load(f)
        for k, node_list in data["nodes"].items():
            lid = int(k)
            for n in node_list:
                kg.nodes[lid].append({"emb": torch.tensor(n["emb"]), "text": n["text"]})
        for k, edges in data["intra_edges"].items():
            kg.intra_edges[int(k)] = [tuple(e) for e in edges]
        kg.cross_edges = [tuple(e) for e in data["cross_edges"]]
        print(f"  KG loaded from {path}")
        return kg


# ═══════════════════════════════════════════════════════════
# LAYER 2: KG-RAG RETRIEVAL
# ═══════════════════════════════════════════════════════════
class KGRetriever:
    """
    Layer 2: Retrieve KG triples/facts for a query embedding.
    Returns both (emb, weight) pairs for fusion AND text facts for LLaMA.
    """
    def __init__(self, kg, top_k=KG_TOP_K, top_nodes=KG_TOP_NODES, hop=KG_HOP):
        self.kg        = kg
        self.top_k     = top_k
        self.top_nodes = top_nodes
        self.hop       = hop

    def retrieve_embs(self, h_i, rare_ids=None, first_pass_label=None):
        """Returns list of (emb: Tensor(D,), weight: float)."""
        h_norm = F.normalize(h_i.unsqueeze(0), dim=-1)
        scores = {}
        for lid in self.kg.nodes:
            embs = self.kg._get_stacked(lid)
            if embs is None:
                continue
            sims = torch.mv(F.normalize(embs, dim=-1), h_norm.squeeze(0))
            scores[lid] = float(sims.max())
        top_lids = sorted(scores, key=lambda x: -scores[x])[:self.top_k]

        results = []
        for lid in top_lids:
            embs = self.kg._get_stacked(lid)
            if embs is None:
                continue
            sims = torch.mv(F.normalize(embs, dim=-1), h_norm.squeeze(0))
            k    = min(self.top_nodes, embs.shape[0])
            top_idx  = sims.topk(k).indices.tolist()
            top_sims = sims.topk(k).values.tolist()
            seed_set = set(top_idx)

            if self.hop >= 1:
                for idx in list(seed_set):
                    for (i, j, w) in self.kg.intra_edges.get(lid, []):
                        if i == idx and j not in seed_set:
                            seed_set.add(j)
                        elif j == idx and i not in seed_set:
                            seed_set.add(i)
                for (la, ni, lb, nj, w) in self.kg.cross_edges:
                    if la == lid and ni in seed_set:
                        ce = self.kg._get_stacked(lb)
                        if ce is not None and nj < ce.shape[0]:
                            results.append((ce[nj], w))
                    elif lb == lid and nj in seed_set:
                        ce = self.kg._get_stacked(la)
                        if ce is not None and ni < ce.shape[0]:
                            results.append((ce[ni], w))

            for idx, sim in zip(top_idx, top_sims):
                results.append((embs[idx], float(sim)))

        return results

    def retrieve_texts(self, h_i):
        """Returns list of text strings from top retrieved nodes."""
        return self.kg.retrieve_text_facts(h_i, top_k=self.top_k, top_nodes=self.top_nodes)


# ═══════════════════════════════════════════════════════════
# UNCERTAINTY ESTIMATOR
# ═══════════════════════════════════════════════════════════
class UncertaintyEstimator:
    def __init__(self, num_classes=NUM_LABELS, threshold=UNCERTAINTY_THRESH):
        self.log_C     = math.log(num_classes)
        self.threshold = threshold

    def entropy(self, logits):
        probs = F.softmax(logits, dim=-1)
        H = -(probs * (probs + 1e-9).log()).sum(dim=-1)
        return H / self.log_C

    def is_uncertain(self, logits):
        return self.entropy(logits) > self.threshold

    def top_label(self, logits):
        return logits.argmax(dim=-1)


# ═══════════════════════════════════════════════════════════
# LAYER 3: CONTEXT FUSION
# ═══════════════════════════════════════════════════════════
class GraphAttentionFusion(nn.Module):
    """Graph-attention weighted aggregation of KG neighbours."""
    def __init__(self, emb_dim=KG_FUSION_DIM, dropout=DROPOUT):
        super().__init__()
        self.proj_q  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.proj_k  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.scale   = emb_dim ** -0.5

    def forward(self, h_i, neighbours, device=None):
        if not neighbours:
            return torch.zeros_like(h_i)
        if device is None:
            device = h_i.device
        embs    = torch.stack([nb[0] for nb in neighbours]).to(device)
        weights = torch.tensor([nb[1] for nb in neighbours], device=device, dtype=torch.float)
        q    = self.proj_q(h_i.unsqueeze(0))
        k    = self.proj_k(embs)
        dot  = torch.mv(k, q.squeeze(0)) * self.scale
        alpha = F.softmax(dot * weights, dim=0)
        alpha = self.dropout(alpha)
        return (alpha.unsqueeze(-1) * embs).sum(dim=0)


class ContextFusion(nn.Module):
    """
    Layer 3: Fuse [sentence_emb + KG_emb + positional_features].
    Output: fused_dim representation ready for LLaMA.
    """
    def __init__(
        self,
        sent_dim     = KG_FUSION_DIM,   # 256
        pos_dim      = 16,
        fusion_dim   = FUSION_DIM,      # 512
        max_pos      = 512,
        dropout      = DROPOUT,
    ):
        super().__init__()
        self.pos_emb    = nn.Embedding(max_pos, pos_dim)
        self.rel_pos    = nn.Embedding(5, pos_dim)  # relative section encoding

        # KG attention fusion (same dim as sent_dim)
        self.gat = GraphAttentionFusion(emb_dim=sent_dim, dropout=dropout)

        # Project concatenated features → fusion_dim
        in_dim = sent_dim * 2 + pos_dim  # sent + kg_fused + positional
        self.proj = nn.Sequential(
            nn.Linear(in_dim, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # Context BiLSTM over fused sentence sequence
        self.ctx_bilstm = nn.LSTM(
            input_size    = fusion_dim,
            hidden_size   = fusion_dim // 2,
            num_layers    = CTX_LSTM_LAYERS,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if CTX_LSTM_LAYERS > 1 else 0.0,
        )
        self.out_dim = fusion_dim

    def forward(self, sent_vecs, kg_neighbours_batch, positions, lengths, device):
        """
        sent_vecs         : (B, T, sent_dim)
        kg_neighbours_batch : list[list[list[(emb,w)]]]  B x T x K
        positions         : (B, T) long
        lengths           : (B,)
        Returns           : (B, T, fusion_dim)
        """
        B, T, D = sent_vecs.shape

        # KG graph attention fusion per sentence
        kg_vecs = torch.zeros_like(sent_vecs)   # (B, T, D)
        for b in range(B):
            for t in range(int(lengths[b])):
                nbrs = kg_neighbours_batch[b][t]
                if nbrs:
                    kg_vecs[b, t] = self.gat(sent_vecs[b, t], nbrs, device=device)

        # Positional features
        pos_emb = self.pos_emb(positions.clamp(0, 511))  # (B, T, pos_dim)

        # Concatenate
        fused = torch.cat([sent_vecs, kg_vecs, pos_emb], dim=-1)  # (B, T, 2D+pos)
        fused = self.proj(fused)                                    # (B, T, fusion_dim)

        # Context BiLSTM
        packed = nn.utils.rnn.pack_padded_sequence(
            fused, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_out, _ = self.ctx_bilstm(packed)
        ctx_out, _    = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)

        return ctx_out  # (B, T, fusion_dim)


# ═══════════════════════════════════════════════════════════
# LAYER 4: LLaMA LoRA REASONING HEAD
# ═══════════════════════════════════════════════════════════
def build_llama_lora(model_name=LLAMA_MODEL_NAME, use_4bit=USE_4BIT):
    """Load LLaMA with LoRA adapters."""
    print(f"\nLoading LLaMA: {model_name} ...")

    if use_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit              = True,
            bnb_4bit_quant_type       = "nf4",
            bnb_4bit_compute_dtype    = torch.float16,
            bnb_4bit_use_double_quant = True,
        )
        llama = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config = bnb_config,
            device_map          = "auto",
            trust_remote_code   = True,
        )
        llama = prepare_model_for_kbit_training(llama)
    else:
        llama = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype       = torch.float16,
            device_map        = "auto",
            trust_remote_code = True,
        )

    lora_config = LoraConfig(
        task_type    = TaskType.FEATURE_EXTRACTION,
        r            = LORA_R,
        lora_alpha   = LORA_ALPHA,
        lora_dropout = LORA_DROPOUT,
        target_modules = LORA_TARGET,
        bias         = "none",
    )
    llama = get_peft_model(llama, lora_config)
    llama.print_trainable_parameters()
    return llama


class LLaMAReasoningHead(nn.Module):
    """
    Layer 4: Uses LLaMA (LoRA fine-tuned) to reason over the enriched
    representation produced by Layer 3.

    Since LLaMA is a causal LM and we need sequence-level hidden states,
    we:
      1. Project fusion_dim → llama_hidden via a linear adapter.
      2. Prepend the projected vectors as "soft prompt tokens" to the
         LLaMA embedding space.
      3. Run a forward pass in "encoder mode" (no generation):
         get hidden states from the last transformer layer.
      4. Pool the output hidden states per sentence position.
    """
    def __init__(
        self,
        llama_model,
        llama_tokenizer,
        fusion_dim  = FUSION_DIM,
        llama_hidden = LLAMA_HIDDEN,
        dropout     = DROPOUT,
    ):
        super().__init__()
        self.llama     = llama_model
        self.tokenizer = llama_tokenizer
        self.fusion_dim  = fusion_dim
        self.llama_hidden = llama_hidden

        # Adapter: project encoder features → LLaMA embedding space
        self.adapter = nn.Sequential(
            nn.Linear(fusion_dim, llama_hidden),
            nn.LayerNorm(llama_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        # Output projection: llama_hidden → fusion_dim
        self.out_proj = nn.Sequential(
            nn.Linear(llama_hidden, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.out_dim = fusion_dim

    def forward(self, ctx_vecs, kg_text_tokens=None, lengths=None, device=None):
        """
        ctx_vecs       : (B, T, fusion_dim) — output from Layer 3
        kg_text_tokens : optional dict of tokenised KG text (for chain-of-thought)
        lengths        : (B,)
        Returns        : (B, T, fusion_dim) — LLaMA-reasoned representations
        """
        B, T, _ = ctx_vecs.shape
        if device is None:
            device = ctx_vecs.device

        # Project to LLaMA space: (B, T, llama_hidden)
        adapted = self.adapter(ctx_vecs)

        # Flatten to (B*T, 1, llama_hidden) as single-token soft prompts
        # Then run LLaMA's transformer layers to get contextualised hidden states
        # We process each document's sentences as a sequence of soft tokens.
        results = []
        for b in range(B):
            n = int(lengths[b]) if lengths is not None else T
            soft_tokens = adapted[b, :n, :]  # (n, llama_hidden)

            # Feed into LLaMA as inputs_embeds (bypass embedding layer)
            # Use output_hidden_states=True to extract last-layer states
            with torch.cuda.amp.autocast(enabled=(device.type == "cuda"
                                                  if hasattr(device, 'type')
                                                  else False)):
                out = self.llama(
                    inputs_embeds    = soft_tokens.unsqueeze(0).to(torch.float16),
                    output_hidden_states = True,
                    return_dict      = True,
                    use_cache        = False,
                )
            # Last hidden state: (1, n, llama_hidden)
            last_hidden = out.hidden_states[-1].squeeze(0).to(ctx_vecs.dtype)  # (n, H)
            # Project back to fusion_dim
            out_vec = self.out_proj(last_hidden)  # (n, fusion_dim)

            # Pad back to T
            padded = torch.zeros(T, self.out_dim, dtype=ctx_vecs.dtype, device=device)
            padded[:n] = out_vec
            results.append(padded)

        return torch.stack(results, dim=0)  # (B, T, fusion_dim)


# ═══════════════════════════════════════════════════════════
# LAYER 5: CLASSIFIER HEAD  (Linear → Softmax → Label)
# ═══════════════════════════════════════════════════════════
class ClassifierHead(nn.Module):
    """Layer 5: Classifier with CRF for sequence coherence."""
    def __init__(self, in_dim=FUSION_DIM, num_labels=NUM_LABELS, dropout=DROPOUT):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_dim, in_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(in_dim // 2, num_labels),
        )
        self.crf = CRF(num_tags=num_labels, batch_first=True)

    def get_emissions(self, x):
        return self.classifier(x)

    def forward(self, x, labels=None, lengths=None):
        emissions = self.get_emissions(x)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)

        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool, device=emissions.device)

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0
            crf_loss = -self.crf(emissions, safe_labels, mask=mask, reduction="mean")
            return crf_loss, emissions
        else:
            return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# FULL 5-LAYER MODEL
# ═══════════════════════════════════════════════════════════
class LegalRRCModel(nn.Module):
    """
    Full 5-Layer pipeline:
      L1: InLegalBERT encoder (context-windowed)
      L2: KG-RAG retrieval
      L3: Context fusion (sent + KG + positional + BiLSTM)
      L4: LLaMA LoRA reasoning
      L5: Classifier + CRF
    """
    def __init__(
        self,
        encoder:   InLegalBERTEncoder,
        retriever: KGRetriever,
        fusion:    ContextFusion,
        llama_head: LLaMAReasoningHead,
        classifier: ClassifierHead,
        rare_ids:  list,
        imbalance_loss: CombinedImbalanceLoss,
    ):
        super().__init__()
        self.encoder       = encoder
        self.retriever     = retriever
        self.fusion        = fusion
        self.llama_head    = llama_head
        self.classifier    = classifier
        self.rare_ids      = rare_ids
        self.imbalance_loss = imbalance_loss
        self.uncertainty   = UncertaintyEstimator()

        # Quick first-pass classifier (on encoder output, before LLaMA)
        # Used for uncertainty estimation to decide KG retrieval
        self.first_pass_cls = nn.Linear(encoder.sent_out_dim, NUM_LABELS)

    def _collect_kg_neighbours(self, sent_vecs, first_pass_logits, lengths, device):
        """
        Collect KG embedding neighbours for each sentence.
        Only retrieves for uncertain or rare-class sentences.
        Returns: B x T x list[(emb, weight)]
        """
        B, T, D = sent_vecs.shape
        kg_neighbours = [[[] for _ in range(T)] for _ in range(B)]

        uncertain_mask = self.uncertainty.is_uncertain(first_pass_logits)
        top_labels     = self.uncertainty.top_label(first_pass_logits)

        for b in range(B):
            n = int(lengths[b])
            for t in range(n):
                uncertain = bool(uncertain_mask[b, t])
                pred_lbl  = int(top_labels[b, t])
                is_rare   = pred_lbl in self.rare_ids

                if uncertain or (RARE_ALWAYS_KG and is_rare):
                    h_i = sent_vecs[b, t].detach().cpu()
                    nbrs = self.retriever.retrieve_embs(
                        h_i, rare_ids=self.rare_ids, first_pass_label=pred_lbl
                    )
                    kg_neighbours[b][t] = nbrs

        return kg_neighbours

    def forward(
        self,
        input_ids,
        attention_mask,
        token_type_ids,
        labels   = None,
        lengths  = None,
        raw_sents = None,
    ):
        device = input_ids.device
        B, T, _ = input_ids.shape

        # ── Layer 1: Encode sentences ──────────────────────────
        sent_vecs = self.encoder(input_ids, attention_mask, token_type_ids, lengths)
        # (B, T, 256)

        # ── First-pass logits for uncertainty ─────────────────
        fp_logits = self.first_pass_cls(sent_vecs)  # (B, T, C) – no CRF

        # ── Layer 2: KG retrieval ──────────────────────────────
        kg_neighbours = self._collect_kg_neighbours(
            sent_vecs, fp_logits, lengths, device
        )

        # ── Positional features ────────────────────────────────
        positions = torch.arange(T, device=device).unsqueeze(0).expand(B, T)

        # ── Layer 3: Context fusion ────────────────────────────
        ctx_vecs = self.fusion(sent_vecs, kg_neighbours, positions, lengths, device)
        # (B, T, fusion_dim)

        # ── Layer 4: LLaMA reasoning ───────────────────────────
        reasoned = self.llama_head(ctx_vecs, lengths=lengths, device=device)
        # (B, T, fusion_dim)

        # Residual: combine reasoned + ctx for gradient flow
        combined = reasoned + ctx_vecs

        # ── Layer 5: Classify ──────────────────────────────────
        if lengths is not None:
            mask = torch.zeros(B, T, dtype=torch.bool, device=device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(B, T, dtype=torch.bool, device=device)

        emissions = self.classifier.get_emissions(combined)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0

            crf_loss = -self.classifier.crf(
                emissions, safe_labels, mask=mask, reduction="mean"
            )
            # Imbalance-aware auxiliary loss
            B2, T2, C = emissions.shape
            aux_loss = self.imbalance_loss(
                emissions.reshape(B2 * T2, C),
                labels.reshape(B2 * T2),
            )
            # First-pass loss for curriculum signal
            fp_loss = F.cross_entropy(
                fp_logits.reshape(B2*T2, C),
                labels.reshape(B2*T2),
                ignore_index=-100,
            )
            loss = crf_loss + AUX_CE_WEIGHT * aux_loss + 0.1 * fp_loss
            return loss, emissions
        else:
            decoded = self.classifier.crf.decode(emissions, mask=mask)
            return decoded, emissions


# ═══════════════════════════════════════════════════════════
# BASE-ONLY MODEL  (Phase A — no LLaMA)
# ═══════════════════════════════════════════════════════════
class BaseModel(nn.Module):
    """
    Phase A training model: L1 + simplified L3 + L5 (no LLaMA).
    Used to pre-train the encoder and build the KG.
    """
    def __init__(self, encoder, imbalance_loss, dropout=DROPOUT):
        super().__init__()
        self.encoder        = encoder
        self.imbalance_loss = imbalance_loss

        self.ctx_bilstm = nn.LSTM(
            input_size    = encoder.sent_out_dim,
            hidden_size   = CTX_LSTM_HIDDEN,
            num_layers    = CTX_LSTM_LAYERS,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if CTX_LSTM_LAYERS > 1 else 0.0,
        )
        ctx_out_dim = CTX_LSTM_HIDDEN * 2

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(ctx_out_dim, ctx_out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ctx_out_dim // 2, NUM_LABELS),
        )
        self.crf = CRF(num_tags=NUM_LABELS, batch_first=True)

    def get_sent_vecs(self, input_ids, attention_mask, token_type_ids, lengths=None):
        return self.encoder(input_ids, attention_mask, token_type_ids, lengths)

    def forward(self, input_ids, attention_mask, token_type_ids,
                labels=None, lengths=None):
        sent_vecs = self.get_sent_vecs(input_ids, attention_mask, token_type_ids, lengths)

        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sent_vecs, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _    = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        else:
            ctx_out, _ = self.ctx_bilstm(sent_vecs)

        emissions = self.classifier(ctx_out)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)

        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool, device=emissions.device)

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0
            crf_loss = -self.crf(emissions, safe_labels, mask=mask, reduction="mean")
            B2, T2, C = emissions.shape
            aux_loss = self.imbalance_loss(
                emissions.reshape(B2*T2, C), labels.reshape(B2*T2)
            )
            return crf_loss + AUX_CE_WEIGHT * aux_loss, emissions
        else:
            return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# METRICS
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_prec    = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec    = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_prec = precision_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_rec    = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec    = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_rec = recall_score(all_trues, all_preds, average="weighted", zero_division=0)

    acc = accuracy_score(all_trues, all_preds)

    per_class_f1   = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                              average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                     average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                  average=None, zero_division=0)

    per_class_metrics = {
        id2label[i]: {
            "f1": float(per_class_f1[i]),
            "precision": float(per_class_prec[i]),
            "recall": float(per_class_rec[i]),
        }
        for i in range(NUM_LABELS)
    }

    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare, average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare, average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare, average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0

    str_trues = [id2label[x] for x in all_trues]
    str_preds = [id2label[x] for x in all_preds]
    cls_report = classification_report(str_trues, str_preds, labels=LABELS, digits=4, zero_division=0)
    cm = confusion_matrix(str_trues, str_preds, labels=LABELS)

    return {
        "macro_f1": macro_f1, "micro_f1": micro_f1, "weighted_f1": weighted_f1,
        "macro_precision": macro_prec, "micro_precision": micro_prec,
        "weighted_precision": weighted_prec,
        "macro_recall": macro_rec, "micro_recall": micro_rec,
        "weighted_recall": weighted_rec,
        "rare_f1": rare_f1, "rare_precision": rare_prec, "rare_recall": rare_rec,
        "per_class_metrics": per_class_metrics,
        "accuracy": acc, "cls_report": cls_report, "cm": cm,
        "all_preds": all_preds, "all_trues": all_trues,
    }


# ═══════════════════════════════════════════════════════════
# PARAMETER COUNTER
# ═══════════════════════════════════════════════════════════
def count_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"  Trainable: {trainable:,} | Frozen: {frozen:,}")
    return trainable, frozen


# ═══════════════════════════════════════════════════════════
# PHASE A TRAINER  (base model only)
# ═══════════════════════════════════════════════════════════
class BaseTrainer:
    def __init__(self, model: BaseModel, device=DEVICE):
        self.model  = model.to(device)
        self.device = device

    def build_optimizer(self):
        param_groups = []
        encoder = self.model.encoder

        # BERT pooler
        param_groups.append({
            "params": list(encoder.bert.pooler.parameters()),
            "lr": BERT_LR, "weight_decay": WEIGHT_DECAY,
        })
        # Layer-wise LR decay for unfrozen BERT layers
        enc_layers = encoder.bert.encoder.layer
        n = len(enc_layers)
        for i in range(n - 1, BERT_FREEZE_LAYERS - 1, -1):
            depth = (n - 1) - i
            lr_i  = BERT_LR * (BERT_LR_DECAY ** depth)
            params = [p for p in enc_layers[i].parameters() if p.requires_grad]
            if params:
                param_groups.append({"params": params, "lr": lr_i, "weight_decay": WEIGHT_DECAY})

        # Head modules
        head_params = (
            list(encoder.sent_bilstm.parameters()) +
            list(encoder.mha_pooling.parameters()) +
            list(encoder.sent_layer_norm.parameters()) +
            list(self.model.ctx_bilstm.parameters()) +
            list(self.model.classifier.parameters()) +
            list(self.model.crf.parameters())
        )
        param_groups.append({"params": head_params, "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY})
        return torch.optim.AdamW(param_groups)

    def evaluate(self, dataset, rare_ids, split_name="dev"):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        with torch.no_grad():
            for ids, attn, ttype, labels, lengths, _ in loader:
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                lengths = lengths.to(self.device)
                decoded, _ = self.model(ids, attn, ttype, labels=None, lengths=lengths)
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i])
                    all_preds.extend(seq_preds)
                    all_trues.extend(labels[i, :true_len].tolist())
        return compute_all_metrics(all_trues, all_preds, rare_ids, split_name)

    def train(self, train_dataset, dev_dataset, rare_ids,
              tokenizer, num_epochs=NUM_EPOCHS_BASE):
        # Balanced sampling
        sample_weights = compute_sample_weights(train_dataset.docs)
        sampler = WeightedRandomSampler(
            weights     = sample_weights,
            num_samples = len(sample_weights),
            replacement = True,
        )
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                   sampler=sampler, collate_fn=collate_rrc)
        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

        early_stopper = EarlyStopping()
        history = []
        best_f1, best_state = -1.0, None
        total_start = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            running_loss, n_steps = 0.0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for step, (ids, attn, ttype, labels, lengths, _) in enumerate(train_loader):
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                labels  = labels.to(self.device)
                lengths = lengths.to(self.device)

                loss, _ = self.model(ids, attn, ttype, labels=labels, lengths=lengths)
                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue

                (loss / GRADIENT_ACCUMULATION_STEPS).backward()
                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                    optimizer.step(); scheduler.step(); optimizer.zero_grad()

                running_loss += loss.item(); n_steps += 1

            epoch_time = time.time() - epoch_start
            avg_loss   = running_loss / max(1, n_steps)
            val_m      = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[Base] Epoch {epoch:03d}/{num_epochs} | "
                f"loss: {avg_loss:.4f} | val_macro_f1: {val_m['macro_f1']:.4f} | "
                f"val_rare_f1: {val_m['rare_f1']:.4f} | "
                f"time: {epoch_time:.1f}s | ES: {early_stopper.counter}/{early_stopper.patience}"
            )
            history.append({
                "epoch": epoch, "phase": "base",
                "train_loss": avg_loss,
                "val_macro_f1": val_m["macro_f1"], "val_rare_f1": val_m["rare_f1"],
                "val_accuracy": val_m["accuracy"], "epoch_train_time_s": epoch_time,
            })

            if val_m["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_m["macro_f1"]
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f}")

            if early_stopper.step(val_m["macro_f1"]):
                print(f"⏹  Early stopping at epoch {epoch}"); break

        total_time = time.time() - total_start
        pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, "base_history.csv"), index=False)
        if best_state:
            self.model.load_state_dict(best_state)
            tokenizer.save_pretrained(BEST_MODEL_DIR)
            torch.save(best_state, os.path.join(BEST_MODEL_DIR, "base_model.bin"))
            print(f"✔ Base model saved (val_macro_f1={best_f1:.4f})")
        return pd.DataFrame(history), total_time


# ═══════════════════════════════════════════════════════════
# KG BUILDER
# ═══════════════════════════════════════════════════════════
@torch.no_grad()
def build_knowledge_graph(base_model: BaseModel, train_docs, tokenizer,
                           device=DEVICE) -> KnowledgeGraph:
    print("\n🔨 Building Knowledge Graph ...")
    base_model.eval()
    base_model.to(device)
    kg = KnowledgeGraph(emb_dim=base_model.encoder.sent_out_dim)

    dataset = RRCDataset(train_docs, tokenizer)
    loader  = DataLoader(dataset, batch_size=1, shuffle=False, collate_fn=collate_rrc)

    for doc_idx, (ids, attn, ttype, labels, lengths, raw_sents) in enumerate(loader):
        ids     = ids.to(device)
        attn    = attn.to(device)
        ttype   = ttype.to(device)
        sent_vecs = base_model.get_sent_vecs(ids, attn, ttype)
        sent_vecs = sent_vecs.squeeze(0)
        n = int(lengths[0])
        embs    = sent_vecs[:n].cpu()
        lab_ids = labels[0, :n].tolist()
        texts   = raw_sents[0][:n]
        kg.add_nodes(embs, lab_ids, texts=texts)
        if (doc_idx + 1) % 50 == 0:
            print(f"  Processed {doc_idx+1}/{len(loader)} docs")

    kg.build_edges()
    kg_path = os.path.join(OUT_DIR, "knowledge_graph.json")
    kg.save(kg_path)
    return kg


# ═══════════════════════════════════════════════════════════
# PHASE B TRAINER  (full 5-layer model with LLaMA LoRA)
# ═══════════════════════════════════════════════════════════
class FullModelTrainer:
    def __init__(self, model: LegalRRCModel, device=DEVICE):
        self.model  = model.to(device)
        self.device = device

    def build_optimizer(self):
        # LoRA params get higher LR; encoder gets BERT_LR; fusion/classifier HEAD_LR
        lora_params = [p for n, p in self.model.llama_head.named_parameters()
                       if "lora" in n.lower() and p.requires_grad]
        adapter_params = (
            list(self.model.llama_head.adapter.parameters()) +
            list(self.model.llama_head.out_proj.parameters())
        )
        fusion_params = (
            list(self.model.fusion.parameters()) +
            list(self.model.classifier.parameters()) +
            list(self.model.first_pass_cls.parameters())
        )
        encoder_params = [p for p in self.model.encoder.parameters() if p.requires_grad]

        return torch.optim.AdamW([
            {"params": lora_params,    "lr": LORA_LR,  "weight_decay": WEIGHT_DECAY},
            {"params": adapter_params, "lr": HEAD_LR,  "weight_decay": WEIGHT_DECAY},
            {"params": fusion_params,  "lr": HEAD_LR,  "weight_decay": WEIGHT_DECAY},
            {"params": encoder_params, "lr": BERT_LR,  "weight_decay": WEIGHT_DECAY},
        ])

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples = len(dataset)
        t_start = time.time() if measure_inference_time else None

        with torch.no_grad():
            for ids, attn, ttype, labels, lengths, raw_sents in loader:
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                lengths = lengths.to(self.device)
                decoded, _ = self.model(ids, attn, ttype,
                                        labels=None, lengths=lengths, raw_sents=raw_sents)
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i])
                    all_preds.extend(seq_preds)
                    all_trues.extend(labels[i, :true_len].tolist())

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if measure_inference_time:
            total_t = time.time() - t_start
            metrics["inference_time_info"] = {
                "total_inference_time_s": total_t,
                "latency_per_document_ms": total_t / max(1, n_samples) * 1000,
                "throughput_sentences_per_s": len(all_trues) / max(1e-9, total_t),
            }
        return metrics

    def train(self, train_dataset, dev_dataset, rare_ids,
              num_epochs=NUM_EPOCHS_KG):
        sample_weights = compute_sample_weights(train_dataset.docs)
        sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                   sampler=sampler, collate_fn=collate_rrc)

        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

        scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

        early_stopper = EarlyStopping(patience=5)
        history = []
        best_f1, best_state = -1.0, None
        total_start = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            running_loss, n_steps = 0.0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for ids, attn, ttype, labels, lengths, raw_sents in train_loader:
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                labels  = labels.to(self.device)
                lengths = lengths.to(self.device)

                with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                    loss, _ = self.model(ids, attn, ttype,
                                         labels=labels, lengths=lengths, raw_sents=raw_sents)

                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                scaler.step(optimizer); scaler.update()
                scheduler.step(); optimizer.zero_grad()

                running_loss += loss.item(); n_steps += 1

            epoch_time = time.time() - epoch_start
            avg_loss   = running_loss / max(1, n_steps)
            val_m      = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[Full] Epoch {epoch:02d}/{num_epochs} | "
                f"loss: {avg_loss:.4f} | val_macro_f1: {val_m['macro_f1']:.4f} | "
                f"val_rare_f1: {val_m['rare_f1']:.4f} | "
                f"time: {epoch_time:.1f}s | ES: {early_stopper.counter}/{early_stopper.patience}"
            )
            history.append({
                "epoch": epoch, "phase": "full",
                "train_loss": avg_loss,
                "val_macro_f1": val_m["macro_f1"], "val_rare_f1": val_m["rare_f1"],
                "val_accuracy": val_m["accuracy"], "epoch_train_time_s": epoch_time,
            })

            if val_m["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_m["macro_f1"]
                # Save only trainable weights for memory efficiency
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()
                              if any(p is self.model.state_dict()[k]
                                     for p in self.model.parameters() if p.requires_grad)
                              or True}  # save all; LLaMA frozen weights excluded by PEFT
                print(f"  ✔ New best val_macro_f1={best_f1:.4f}")

            if early_stopper.step(val_m["macro_f1"]):
                print(f"⏹  Full model early stopping at epoch {epoch}"); break

        total_time = time.time() - total_start
        pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, "full_history.csv"), index=False)
        if best_state:
            self.model.load_state_dict(best_state, strict=False)
            torch.save(best_state, os.path.join(BEST_MODEL_DIR, "full_model.bin"))
            print(f"✔ Full model saved (val_macro_f1={best_f1:.4f})")
        return pd.DataFrame(history), total_time


# ═══════════════════════════════════════════════════════════
# VISUALISATIONS
# ═══════════════════════════════════════════════════════════
def save_confusion_matrix(cm, split_name, rare_labels=None):
    fig, ax = plt.subplots(figsize=(14, 11))
    sns.heatmap(cm, annot=True, fmt="d",
                xticklabels=LABELS, yticklabels=LABELS, cmap="Blues", ax=ax)
    if rare_labels:
        for tick in ax.get_xticklabels():
            if tick.get_text() in rare_labels:
                tick.set_color("red")
        for tick in ax.get_yticklabels():
            if tick.get_text() in rare_labels:
                tick.set_color("red")
    ax.set_title(f"{split_name.capitalize()} Confusion Matrix")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"Saved {path}")


def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
    f1s    = [per_class_metrics[l]["f1"] for l in LABELS]
    colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue" for l in LABELS]
    fig, ax = plt.subplots(figsize=(9, 6))
    bars = ax.barh(LABELS, f1s, color=colors, edgecolor="white")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_xlim(0, 1.12)
    ax.set_xlabel("F1 Score")
    ax.set_title(f"{split_name.capitalize()} Per-Class F1 (KG-RAG + LLaMA LoRA)")
    ax.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"Saved {path}")


def plot_combined_history(base_df, full_df):
    all_df = pd.concat([base_df, full_df], ignore_index=True)
    all_df["global_epoch"] = range(1, len(all_df) + 1)
    boundary = len(base_df)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(all_df["global_epoch"], all_df["train_loss"], marker="o", markersize=3)
    axes[0].axvline(boundary, color="red", ls="--", label="Phase B start")
    axes[0].set_title("Training Loss"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(all_df["global_epoch"], all_df["val_macro_f1"],
                 label="Val Macro-F1", marker="o", markersize=3)
    axes[1].plot(all_df["global_epoch"], all_df["val_rare_f1"],
                 label="Val Rare-F1", marker="s", markersize=3)
    axes[1].axvline(boundary, color="red", ls="--", label="Phase B start")
    axes[1].set_title("Val F1"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(OUT_DIR, "combined_training_curves.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"Saved {path}")


def print_metrics_table(dev_m, test_m, base_time=None, full_time=None, trainable=None):
    rows = [
        ("Accuracy",           "accuracy"),
        ("Macro-F1",           "macro_f1"),
        ("Micro-F1",           "micro_f1"),
        ("Weighted-F1",        "weighted_f1"),
        ("Minority Macro-F1",  "rare_f1"),
        ("Macro-Precision",    "macro_precision"),
        ("Macro-Recall",       "macro_recall"),
    ]
    print("\n" + "=" * 72)
    print("FINAL RESULTS  (InLegalBERT + KG-RAG + LLaMA LoRA + CRF)")
    print("=" * 72)
    if trainable:
        print(f"  Trainable Parameters : {trainable:,}")
    if base_time:
        print(f"  Phase A time         : {base_time/60:.1f} min")
    if full_time:
        print(f"  Phase B time         : {full_time/60:.1f} min")
    print("-" * 72)
    print(f"  {'Metric':<28} {'Dev':>12} {'Test':>12}")
    print("-" * 72)
    for label, key in rows:
        print(f"  {label:<28} {dev_m[key]:>12.4f} {test_m[key]:>12.4f}")
    print("=" * 72)

    print("\n  PER-CLASS F1 / PRECISION / RECALL")
    print("  " + "-" * 64)
    print(f"  {'Label':<22} {'F1-Dev':>9} {'F1-Test':>9} {'Prec-Test':>11} {'Rec-Test':>10}")
    print("  " + "-" * 64)
    for lbl in LABELS:
        dv = dev_m["per_class_metrics"][lbl]
        ts = test_m["per_class_metrics"][lbl]
        print(f"  {lbl:<22} {dv['f1']:>9.4f} {ts['f1']:>9.4f} "
              f"{ts['precision']:>11.4f} {ts['recall']:>10.4f}")
    print("  " + "-" * 64)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device: {DEVICE}")
    print("Architecture: InLegalBERT [L1] → KG-RAG [L2] → Context Fusion [L3]")
    print("              → LLaMA LoRA [L4] → Classifier+CRF [L5]\n")

    # ── Data ─────────────────────────────────────────────────
    print("Loading data ...")
    train_docs = extract_docs(load_jsonl(TRAIN_PATH))
    dev_docs   = extract_docs(load_jsonl(DEV_PATH))
    test_docs  = extract_docs(load_jsonl(TEST_PATH))
    print(f"  Train: {len(train_docs)} | Dev: {len(dev_docs)} | Test: {len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)

    pd.DataFrame([{"label": l, "frequency": label_freqs[l], "is_rare": l in rare_labels}
                  for l in LABELS]).to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    # Class imbalance loss components
    all_ids    = [lid for _, labs in train_docs for lid in labs]
    counts     = Counter(all_ids)
    cls_num_list = np.array([counts.get(i, 1) for i in range(NUM_LABELS)], dtype=np.float32)
    class_weights = compute_class_weights(train_docs, rare_ids)

    imbalance_loss = CombinedImbalanceLoss(cls_num_list, class_weights)

    # Tokeniser
    print("Loading InLegalBERT tokenizer ...")
    tokenizer = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)

    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    # ══════════════════════════════════════════════════════
    # PHASE A: Train base encoder
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A: Base Encoder Training (L1 + L3-slim + L5)")
    print("=" * 60)

    encoder    = InLegalBERTEncoder(freeze_layers=BERT_FREEZE_LAYERS)
    base_model = BaseModel(encoder, imbalance_loss)

    base_trainer = BaseTrainer(base_model, device=DEVICE)
    base_hist, base_time = base_trainer.train(
        train_dataset, dev_dataset, rare_ids,
        tokenizer=tokenizer, num_epochs=NUM_EPOCHS_BASE,
    )

    # ══════════════════════════════════════════════════════
    # PHASE A→B: Build Knowledge Graph
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A→B: Building Knowledge Graph")
    print("=" * 60)

    kg_path = os.path.join(OUT_DIR, "knowledge_graph.json")
    if os.path.exists(kg_path):
        kg = KnowledgeGraph.load(kg_path, emb_dim=encoder.sent_out_dim)
    else:
        kg = build_knowledge_graph(base_model, train_docs, tokenizer, DEVICE)

    retriever = KGRetriever(kg, top_k=KG_TOP_K, top_nodes=KG_TOP_NODES, hop=KG_HOP)

    # ══════════════════════════════════════════════════════
    # PHASE B: Assemble and fine-tune full 5-layer model
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE B: Full Model Fine-Tuning (L1→L5 with LLaMA LoRA)")
    print("=" * 60)

    # Layer 3: Context Fusion
    fusion = ContextFusion(
        sent_dim   = encoder.sent_out_dim,
        fusion_dim = FUSION_DIM,
        dropout    = DROPOUT,
    )

    # Layer 4: LLaMA LoRA
    llama_model = build_llama_lora(model_name=LLAMA_MODEL_NAME, use_4bit=USE_4BIT)
    llama_tokenizer = LLMTokenizer.from_pretrained(LLAMA_MODEL_NAME)
    if llama_tokenizer.pad_token is None:
        llama_tokenizer.pad_token = llama_tokenizer.eos_token

    llama_head = LLaMAReasoningHead(
        llama_model     = llama_model,
        llama_tokenizer = llama_tokenizer,
        fusion_dim      = FUSION_DIM,
        llama_hidden    = LLAMA_HIDDEN,
        dropout         = DROPOUT,
    )

    # Layer 5: Classifier + CRF
    classifier = ClassifierHead(in_dim=FUSION_DIM, num_labels=NUM_LABELS, dropout=DROPOUT)

    # Full model
    full_model = LegalRRCModel(
        encoder        = encoder,
        retriever      = retriever,
        fusion         = fusion,
        llama_head     = llama_head,
        classifier     = classifier,
        rare_ids       = rare_ids,
        imbalance_loss = imbalance_loss,
    )

    total_trainable, _ = count_parameters(full_model)

    full_trainer = FullModelTrainer(full_model, device=DEVICE)
    full_hist, full_time = full_trainer.train(
        train_dataset, dev_dataset, rare_ids, num_epochs=NUM_EPOCHS_KG,
    )

    plot_combined_history(base_hist, full_hist)

    # ══════════════════════════════════════════════════════
    # EVALUATION
    # ══════════════════════════════════════════════════════
    print("\nEvaluating on Dev ...")
    dev_m = full_trainer.evaluate(dev_dataset, rare_ids, split_name="dev",
                                   measure_inference_time=True)
    print(f"  Dev  Accuracy : {dev_m['accuracy']:.4f}")
    print(f"  Dev  Macro-F1 : {dev_m['macro_f1']:.4f}")
    print(f"  Dev  Rare-F1  : {dev_m['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "dev_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT+KG-RAG+LLaMA-LoRA+CRF\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(dev_m["cls_report"])

    save_confusion_matrix(dev_m["cm"], "dev", rare_labels)
    save_per_class_f1_chart(dev_m["per_class_metrics"], "dev", rare_labels)

    print("\nEvaluating on Test ...")
    test_m = full_trainer.evaluate(test_dataset, rare_ids, split_name="test",
                                    measure_inference_time=True)
    print(f"  Test Accuracy : {test_m['accuracy']:.4f}")
    print(f"  Test Macro-F1 : {test_m['macro_f1']:.4f}")
    print(f"  Test Rare-F1  : {test_m['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "test_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT+KG-RAG+LLaMA-LoRA+CRF\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(test_m["cls_report"])

    save_confusion_matrix(test_m["cm"], "test", rare_labels)
    save_per_class_f1_chart(test_m["per_class_metrics"], "test", rare_labels)

    pd.DataFrame({
        "true": [id2label[x] for x in test_m["all_trues"]],
        "pred": [id2label[x] for x in test_m["all_preds"]],
    }).to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    scalar_keys = [
        "macro_f1", "micro_f1", "weighted_f1", "rare_f1",
        "macro_precision", "micro_precision", "weighted_precision", "rare_precision",
        "macro_recall", "micro_recall", "weighted_recall", "rare_recall", "accuracy",
    ]
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump({
            "model": "InLegalBERT+BiLSTM+MHA+KG-RAG+LLaMA-LoRA+CRF",
            "architecture": {
                "L1": "InLegalBERT (context-windowed) + BiLSTM + MHA Pooling",
                "L2": "KG-RAG Retrieval (RST-proxy edges, uncertainty-gated)",
                "L3": "Context Fusion (sent + KG GAT + positional + BiLSTM)",
                "L4": "LLaMA LoRA (QLoRA 4-bit, soft-prompt forwarding)",
                "L5": "Linear → CRF → Rhetorical Role Label",
            },
            "imbalance_handling": [
                "LDAM Loss", "Focal Loss", "Class-weighted CE",
                "WeightedRandomSampler", "Rare-always-KG retrieval",
                "Label smoothing", "Effective number class weights",
            ],
            "kg_config": {
                "top_k": KG_TOP_K, "top_nodes": KG_TOP_NODES, "hop": KG_HOP,
                "uncertainty_thresh": UNCERTAINTY_THRESH,
            },
            "lora_config": {
                "r": LORA_R, "alpha": LORA_ALPHA, "target": LORA_TARGET,
            },
            "timing": {"phase_a_s": base_time, "phase_b_s": full_time},
            "rare_classes": rare_labels,
            "dev":  {k: dev_m[k]  for k in scalar_keys},
            "test": {k: test_m[k] for k in scalar_keys},
            "per_class_dev":  dev_m["per_class_metrics"],
            "per_class_test": test_m["per_class_metrics"],
        }, f, indent=2)

    print_metrics_table(dev_m, test_m, base_time, full_time, total_trainable)
    print(f"\n📁 All outputs saved to: {OUT_DIR}/")


if __name__ == "__main__":
    main()

Device: cuda:0
Architecture: InLegalBERT [L1] → KG-RAG [L2] → Context Fusion [L3]
              → LLaMA LoRA [L4] → Classifier+CRF [L5]

Loading data ...
  Train: 245 | Dev: 30 | Test: 50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RESPONDENT        2.43%  (  698 samples) ← RARE
   ANALYSIS             36.66%  (10537 samples)
   STA                   1.67%  (  481 samples) ← RARE
   PRE_RELIED            4.97%  ( 1427 samples) ← RARE
   PRE_NOT_RELIED        0.55%  (  158 samples) ← RARE
   RATIO                 2.30%  (  661 samples) ← RARE
   RPC                   3.67%  ( 1055 samples) ← RARE
   NONE                  4.79%  ( 1377 samples) ← RARE

   Rare classes (10): ['RLC', 'ISSUE', 'ARG_PETITIONER', 'ARG_RESPONDEN

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


❄️  BERT frozen: embeddings + layers 0-7
🔥 BERT trainable: layers 8-11 + pooler



In [2]:
!python -m pip install peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 5.0 MB/s  0:00:00
